## Demo Notebook

Testing the functionality of the Rocchio - SVM classifier

Contains duplicate code for the time being, which will be replaced soon

In [1]:
import pandas as pd

from utils.download import *


download_from_gdrive("deceptive-opinion.csv")

documents = pd.read_csv("../data/deceptive-opinion.csv")
documents.head()

[~_o] File already exists --> Aborting download


,deceptive,hotel,polarity,source,text
0,truthful,conrad,positive,TripAdvisor,We stayed for a one night getaway with family ...
1,truthful,hyatt,positive,TripAdvisor,Triple A rate with upgrade to view room was le...
2,truthful,hyatt,positive,TripAdvisor,This comes a little late as I'm finally catchi...
3,truthful,omni,positive,TripAdvisor,The Omni Chicago really delivers on all fronts...
4,truthful,hyatt,positive,TripAdvisor,I asked for a high floor away from the elevato...


In [2]:
documents = documents.drop(columns=["hotel", "source"])
documents["deceptive"] = documents["deceptive"].apply(lambda x: 0 if x == "truthful" else 1)
documents.head()

,deceptive,polarity,text
0,0,positive,We stayed for a one night getaway with family ...
1,0,positive,Triple A rate with upgrade to view room was le...
2,0,positive,This comes a little late as I'm finally catchi...
3,0,positive,The Omni Chicago really delivers on all fronts...
4,0,positive,I asked for a high floor away from the elevato...


In [3]:
from pulearn.pubase import *


# Get sample positive indices
positive_indices = extract_sample(documents["deceptive"], ratio=0.45)
# Initialize new column with "unlabeled" values
documents["pu-label"] = 0
# Mark the selected positive data points
documents.loc[positive_indices, "pu-label"] = 1

documents

,deceptive,polarity,text,pu-label
0,0,positive,We stayed for a one night getaway with family ...,0
1,0,positive,Triple A rate with upgrade to view room was le...,0
2,0,positive,This comes a little late as I'm finally catchi...,0
3,0,positive,The Omni Chicago really delivers on all fronts...,0
4,0,positive,I asked for a high floor away from the elevato...,0
...,...,...,...,...
1595,1,negative,Problems started when I booked the InterContin...,0
1596,1,negative,The Amalfi Hotel has a beautiful website and i...,0
1597,1,negative,The Intercontinental Chicago Magnificent Mile ...,0
1598,1,negative,"The Palmer House Hilton, while it looks good i...",1


In [4]:
import nltk, re

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download('omw-1.4')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def prepare(row):
    filtered = []
    row = row.lower().strip()
    row = re.sub('[^A-Za-z0-9.]+', ' ', row)
    row_parts = row.split()
    for part in row_parts:
        if part not in stop_words:
            filtered.append(lemmatizer.lemmatize(part))
    return " ".join(filtered)

[nltk_data] Downloading package stopwords to /home/nick/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/nick/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to /home/nick/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Create TF-IDF vectors

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

X = documents["text"]
X = X.apply(lambda row: prepare(row))
y = documents["pu-label"]

vectorizer = TfidfVectorizer()

vectors = vectorizer.fit_transform(X)

feature_names = vectorizer.get_feature_names_out()

dense = vectors.todense().tolist()

X = pd.DataFrame(dense, columns=feature_names)
X

,00,000,00a,00am,00pm,03,04,05,06,07,...,yum,yummo,yummy,yunan,yup,zagat,zest,zipped,zone,zoo
0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1596,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1597,0.116836,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1598,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Use above vectors to train model

The model used below is currently functional, but stil incomplete

In [6]:
from pulearn.iterative_classifiers import RocSVM


roc_svm = RocSVM()
roc_svm.fit(X, y)

Iteration: 0
Unlabeled points remaining: 277

Iteration: 1
Unlabeled points remaining: 115

Iteration: 2
Unlabeled points remaining: 16

Iteration: 3
Unlabeled points remaining: 1

Negative/Positive ratio: 0.09422492401215805
